# PDLP vs Standard LP Solver Comparison

This notebook compares the PDLP (Primal-Dual Linear Programming) first-order solver against a standard LP solver (SciPy's `linprog`).

## Problem Formulation

PDLP solves:
```
minimize    c^T x
subject to  G x >= h   (inequality constraints)
            A x  = b   (equality constraints)
            l <= x <= u (box constraints)
```

SciPy's `linprog` solves:
```
minimize    c^T x
subject to  A_ub x <= b_ub
            A_eq x = b_eq
            l <= x <= u
```

In [18]:
import numpy as np
import scipy.sparse as sp
import scipy.optimize as opt
import time
import pandas as pd
import sys
import os
import glob

# Try to find the pdlp extension in the build folder
build_dir = None
pdlp_so_path = None

# Search for pdlp*.so in build directory and subdirectories
for root, dirs, files in os.walk('build'):
    for f in files:
        if f.startswith('pdlp') and f.endswith(('.so', '.pyd')):
            pdlp_so_path = os.path.join(root, f)
            build_dir = root
            break
    if pdlp_so_path:
        break

if build_dir and build_dir not in sys.path:
    sys.path.insert(0, build_dir)
    if pdlp_so_path:
        print(f"Found pdlp extension at: {pdlp_so_path}")
        print(f"Added {build_dir} to sys.path")
    else:
        print(f"Added build directory to sys.path: {build_dir}")

# Try to import pdlp extension
PDLP_AVAILABLE = False
pdlp_module = None

# First, remove any previously imported pdlp to avoid conflicts
if 'pdlp' in sys.modules:
    del sys.modules['pdlp']

try:
    import pdlp
    if hasattr(pdlp, 'solve'):
        PDLP_AVAILABLE = True
        pdlp_module = pdlp
        print(f"Successfully imported pdlp from: {pdlp.__file__}")
    else:
        print("WARNING: pdlp module imported but has no 'solve' attribute.")
        print("This likely means a different 'pdlp' package is in sys.path.")
except ImportError as e:
    print(f"PDLP extension not available: {e}")
except AttributeError as e:
    print(f"PDLP extension has attribute error: {e}")

print(f"PDLP available: {PDLP_AVAILABLE}")

Successfully imported pdlp from: /data/dev/solvers/build/pdlp.cpython-314-x86_64-linux-gnu.so
PDLP available: True


## Helper Functions

In [19]:
def solve_with_pdlp(c, G, h, A, b, l, u, iteration_limit=10000, verbose=False):
    """Solve LP using PDLP solver."""
    if not PDLP_AVAILABLE or pdlp_module is None or not hasattr(pdlp_module, 'solve'):
        raise RuntimeError("PDLP extension not available or 'solve' function not found. Ensure local pdlp_ext is built and in sys.path.")
    
    start_time = time.perf_counter()
    x, y, status, info = pdlp_module.solve(
        c, G, h, A, b, l, u,
        iteration_limit=iteration_limit,
        eps_tol=1e-6,
        verbose=verbose
    )
    solve_time = time.perf_counter() - start_time
    
    # Convert PyTorch tensors or nanobind vectors to numpy arrays
    def to_numpy(arr):
        if hasattr(arr, 'numpy'):
            return arr.numpy()
        elif hasattr(arr, 'to_numpy'):
            return arr.to_numpy()
        elif isinstance(arr, list):
            return np.array(arr)
        return np.asarray(arr)
    
    return {
        'x': to_numpy(x),
        'y': to_numpy(y),
        'status': status,
        'iterations': info.get('iterations', 0),
        'solve_time_sec': solve_time,
        'primal_obj': info.get('primal_obj', np.nan),
        'dual_obj': info.get('dual_obj', np.nan),
        'duality_gap': info.get('duality_gap', np.nan),
        'primal_residual': info.get('primal_residual', np.nan),
        'dual_residual': info.get('dual_residual', np.nan)
    }


def solve_with_scipy(c, A_ub, b_ub, A_eq, b_eq, bounds, method='highs'):
    """Solve LP using SciPy's linprog."""
    start_time = time.perf_counter()
    
    res = opt.linprog(
        c, 
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds,
        method=method
    )
    solve_time = time.perf_counter() - start_time
    
    return {
        'x': res.x if res.success else np.full_like(c, np.nan),
        'status': res.message,
        'solve_time_sec': solve_time,
        'primal_obj': res.fun if res.success else np.nan,
        'success': res.success
    }


def compute_primal_residual(c, G, h, A, b, l, u, x):
    """Compute primal feasibility residual."""
    # G x >= h
    if G is not None and G.shape[0] > 0:
        Gx = G.dot(x)
        primal_ineq_res = np.maximum(0, h - Gx)
        max_ineq_res = np.max(primal_ineq_res) if len(primal_ineq_res) > 0 else 0.0
    else:
        max_ineq_res = 0.0
    
    # A x = b
    if A is not None and A.shape[0] > 0:
        Ax = A.dot(x)
        eq_res = np.abs(b - Ax)
        max_eq_res = np.max(eq_res) if len(eq_res) > 0 else 0.0
    else:
        max_eq_res = 0.0
    
    # Box constraints
    box_res = np.maximum(0, l - x) + np.maximum(0, x - u)
    max_box_res = np.max(box_res) if len(box_res) > 0 else 0.0
    
    return max(max_ineq_res, max_eq_res, max_box_res)


def compute_dual_residual(c, G, h, A, b, l, u, x, y):
    """Compute dual feasibility residual."""
    # g = c - G^T y_ineq - A^T y_eq
    m1 = G.shape[0] if G is not None and G.shape[0] > 0 else 0
    m2 = A.shape[0] if A is not None and A.shape[0] > 0 else 0
    
    y_ineq = y[:m1] if m1 > 0 else np.array([])
    y_eq = y[m1:] if m2 > 0 else np.array([])
    
    Gt = G.T if G is not None and G.shape[0] > 0 else sp.csr_matrix((0, G.shape[1]))
    At = A.T if A is not None and A.shape[0] > 0 else sp.csr_matrix((0, A.shape[1]))
    
    Gt_y = Gt.dot(y_ineq) if m1 > 0 else np.zeros(G.shape[1])
    At_y = At.dot(y_eq) if m2 > 0 else np.zeros(A.shape[1])
    
    g = c - Gt_y - At_y
    
    # lambda for box constraints
    lam = np.zeros_like(x)
    for i in range(len(x)):
        at_l = np.isfinite(l[i]) and x[i] <= l[i] + 1e-6
        at_u = np.isfinite(u[i]) and x[i] >= u[i] - 1e-6
        if at_l and at_u:
            closer_l = abs(x[i] - l[i]) <= abs(u[i] - x[i])
            at_l = closer_l
            at_u = not closer_l
        
        if at_l:
            lam[i] = max(g[i], 0.0)
        elif at_u:
            lam[i] = min(g[i], 0.0)
    
    return np.linalg.norm(g - lam)

## Test Problem 1: Small LP Problem

In [20]:
# Problem: minimize c^T x subject to Gx >= h, Ax = b, l <= x <= u
# c^T x = -3x1 - 2x2 - x3
# Gx >= h: x1 + x2 + x3 <= 4, x1 - x2 <= 1, x2 - x3 <= 2
# Transform to Gx >= h: -x1 - x2 - x3 >= -4, -x1 + x2 >= -1, -x2 + x3 >= -2
# Ax = b: x1 + x2 + 2x3 = 5
# l <= x <= u: 0 <= x1, x2, x3 <= 10

c = np.array([-3.0, -2.0, -1.0])

# Gx >= h (inequalities)
G = sp.csc_matrix([
    [-1.0, -1.0, -1.0],
    [-1.0,  1.0,  0.0],
    [ 0.0, -1.0,  1.0]
])
h = np.array([-4.0, -1.0, -2.0])

# Ax = b (equalities)
A = sp.csc_matrix([[1.0, 1.0, 2.0]])
b = np.array([5.0])

# Box constraints
l = np.array([0.0, 0.0, 0.0])
u = np.array([10.0, 10.0, 10.0])

print(f"Problem dimensions: n={len(c)}, m1={G.shape[0]} inequalities, m2={A.shape[0]} equalities")
print(f"c = {c}")
print(f"G shape: {G.shape}, h shape: {h.shape}")
print(f"A shape: {A.shape}, b shape: {b.shape}")

Problem dimensions: n=3, m1=3 inequalities, m2=1 equalities
c = [-3. -2. -1.]
G shape: (3, 3), h shape: (3,)
A shape: (1, 3), b shape: (1,)


In [21]:
# Solve with PDLP
if PDLP_AVAILABLE:
    pdlp_result = solve_with_pdlp(c, G, h, A, b, l, u, iteration_limit=10000, verbose=True)
    print(f"\nPDLP Results:")
    print(f"  Status: {pdlp_result['status']}")
    print(f"  Iterations: {pdlp_result['iterations']}")
    print(f"  Solve time: {pdlp_result['solve_time_sec']:.6f}s")
    print(f"  Primal obj: {pdlp_result['primal_obj']:.6f}")
    print(f"  Dual obj: {pdlp_result['dual_obj']:.6f}")
    print(f"  Duality gap: {pdlp_result['duality_gap']:.6e}")
    print(f"  Primal residual: {pdlp_result['primal_residual']:.6e}")
    print(f"  Dual residual: {pdlp_result['dual_residual']:.6e}")
    print(f"  Solution x: {pdlp_result['x']}")
else:
    print("PDLP not available, skipping...")

# Solve with SciPy
# Convert Gx >= h to -Gx <= -h for scipy
A_ub = -G
b_ub = -h

scipy_result = solve_with_scipy(c, A_ub, b_ub, A.toarray(), b, list(zip(l, u)), method='highs')
print(f"\nSciPy Results:")
print(f"  Status: {scipy_result['status']}")
print(f"  Solve time: {scipy_result['solve_time_sec']:.6f}s")
print(f"  Primal obj: {scipy_result['primal_obj']:.6f}")
print(f"  Solution x: {scipy_result['x']}")


PDLP Solver
  Problem: 3 inequalities, 1 equalities, 3 variables
  Rescaling: Ruiz iters=10, Pock-Chambolle alpha=1

PDLP Results:
  Status: optimal
  Iterations: 500
  Solve time: 0.000301s
  Primal obj: -9.000007
  Dual obj: -8.999999
  Duality gap: 8.277032e-06
  Primal residual: 1.943785e-06
  Dual residual: 3.539439e-06
  Solution x: [2.00000199 1.00000201 0.99999726]
  Iter     1: primal_obj = -6.414387e+00, dual_obj = -2.534124e+00, gap = 3.880e+00, KKT = 5.535e+00
  Iter     2: primal_obj = -1.539253e+01, dual_obj = -1.817341e+01, gap = 2.781e+00, KKT = 9.371e+00
  Iter     3: primal_obj = -1.116849e+01, dual_obj = -1.243730e+01, gap = 1.269e+00, KKT = 4.839e+00
  Iter     4: primal_obj = -1.057566e+01, dual_obj = -1.181183e+01, gap = 1.236e+00, KKT = 1.304e+00
  Iter     5: primal_obj = -1.038076e+01, dual_obj = -1.150180e+01, gap = 1.121e+00, KKT = 1.174e+00
  Iter     6: primal_obj = -1.015801e+01, dual_obj = -1.113950e+01, gap = 9.815e-01, KKT = 1.013e+00
  Iter     7: pri

## Test Problem 2: Larger Random LP Problem

In [22]:
def generate_random_lp(n_vars, n_ineq, n_eq, seed=42):
    """Generate a random LP problem."""
    np.random.seed(seed)
    
    c = np.random.randn(n_vars)
    
    # Gx >= h
    G = sp.random(n_ineq, n_vars, density=0.3, format='csc', random_state=seed)
    h = np.random.randn(n_ineq)
    
    # Ax = b
    A = sp.random(n_eq, n_vars, density=0.5, format='csc', random_state=seed)
    b = np.random.randn(n_eq)
    
    # Box constraints
    l = np.zeros(n_vars)
    u = np.full(n_vars, 10.0)
    
    return c, G, h, A, b, l, u

n_vars = 100
n_ineq = 150
n_eq = 30

c, G, h, A, b, l, u = generate_random_lp(n_vars, n_ineq, n_eq, seed=42)
print(f"Generated problem: n={n_vars} vars, m1={n_ineq} inequalities, m2={n_eq} equalities")

Generated problem: n=100 vars, m1=150 inequalities, m2=30 equalities


In [23]:
# Solve with PDLP
if PDLP_AVAILABLE:
    pdlp_result_large = solve_with_pdlp(c, G, h, A, b, l, u, iteration_limit=50000, verbose=False)
    print(f"PDLP Results:")
    print(f"  Status: {pdlp_result_large['status']}")
    print(f"  Iterations: {pdlp_result_large['iterations']}")
    print(f"  Solve time: {pdlp_result_large['solve_time_sec']:.6f}s")
    print(f"  Primal obj: {pdlp_result_large['primal_obj']:.6f}")
    print(f"  Dual obj: {pdlp_result_large['dual_obj']:.6f}")
    print(f"  Duality gap: {pdlp_result_large['duality_gap']:.6e}")
    print(f"  Primal residual: {pdlp_result_large['primal_residual']:.6e}")
    print(f"  Dual residual: {pdlp_result_large['dual_residual']:.6e}")

# Solve with SciPy
A_ub = -G
b_ub = -h

scipy_result_large = solve_with_scipy(c, A_ub, b_ub, A.toarray(), b, list(zip(l, u)), method='highs')
print(f"\nSciPy Results:")
print(f"  Status: {scipy_result_large['status']}")
print(f"  Solve time: {scipy_result_large['solve_time_sec']:.6f}s")
print(f"  Primal obj: {scipy_result_large['primal_obj']:.6f}")

PDLP Results:
  Status: iteration_limit
  Iterations: 50000
  Solve time: 0.624781s
  Primal obj: -2.299150
  Dual obj: 174806061363628507838041686794028168449711827964939145221700865250052903394851943059757391029085109400171848035435141519057933809499010300014533541888.000000
  Duality gap: 1.748061e+149
  Primal residual: 9.519676e+00
  Dual residual: 9.300729e+142

SciPy Results:
  Status: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
  Solve time: 0.001684s
  Primal obj: nan


## Test Problem 3: Unbounded Problem (Dual Infeasible)

In [24]:
# Unbounded problem: minimize -x subject to x >= 0, no upper bound
# Should be dual infeasible (primal unbounded)

c_unbounded = np.array([-1.0])
G_unbounded = sp.csc_matrix([[1.0]])  # x >= 0
h_unbounded = np.array([0.0])
A_unbounded = sp.csc_matrix((0, 1))  # no equalities
b_unbounded = np.array([])
l_unbounded = np.array([0.0])
u_unbounded = np.array([np.inf])

print("Unbounded problem: minimize -x subject to x >= 0")

if PDLP_AVAILABLE:
    pdlp_unbounded = solve_with_pdlp(c_unbounded, G_unbounded, h_unbounded, A_unbounded, b_unbounded, l_unbounded, u_unbounded, iteration_limit=10000)
    print(f"PDLP Status: {pdlp_unbounded['status']}")
    print(f"PDLP Ray: {pdlp_unbounded.get('ray', 'N/A')}")

A_ub_unbounded = -G_unbounded
b_ub_unbounded = -h_unbounded
scipy_unbounded = solve_with_scipy(c_unbounded, A_ub_unbounded, b_ub_unbounded, None, None, [(0, np.inf)], method='highs')
print(f"SciPy Status: {scipy_unbounded['status']}")

Unbounded problem: minimize -x subject to x >= 0
PDLP Status: dual_infeasible
PDLP Ray: N/A
SciPy Status: The problem is unbounded. (HiGHS Status 10: model_status is Unbounded; primal_status is Feasible)


## Test Problem 4: Infeasible Problem (Primal Infeasible)

In [25]:
# Infeasible problem: minimize x subject to x >= 1 and x <= 0
# Should be primal infeasible (dual unbounded)

c_infeasible = np.array([1.0])
G_infeasible = sp.csc_matrix([[1.0], [-1.0]])  # x >= 1, -x >= 0 => x <= 0
h_infeasible = np.array([1.0, 0.0])
A_infeasible = sp.csc_matrix((0, 1))
b_infeasible = np.array([])
l_infeasible = np.array([0.0])
u_infeasible = np.array([np.inf])

print("Infeasible problem: minimize x subject to x >= 1 and x <= 0")

if PDLP_AVAILABLE:
    pdlp_infeasible = solve_with_pdlp(c_infeasible, G_infeasible, h_infeasible, A_infeasible, b_infeasible, l_infeasible, u_infeasible, iteration_limit=10000)
    print(f"PDLP Status: {pdlp_infeasible['status']}")
    print(f"PDLP Ray: {pdlp_infeasible.get('ray', 'N/A')}")

A_ub_infeasible = -G_infeasible
b_ub_infeasible = -h_infeasible
scipy_infeasible = solve_with_scipy(c_infeasible, A_ub_infeasible, b_ub_infeasible, None, None, [(0, np.inf)], method='highs')
print(f"SciPy Status: {scipy_infeasible['status']}")

Infeasible problem: minimize x subject to x >= 1 and x <= 0
PDLP Status: iteration_limit
PDLP Ray: N/A
SciPy Status: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)


## Summary and Comparison

In [26]:
# Create a summary table
results = []

if PDLP_AVAILABLE and 'pdlp_result' in locals():
    results.append({
        'Problem': 'Small LP',
        'Solver': 'PDLP',
        'Status': pdlp_result['status'],
        'Iterations': pdlp_result['iterations'],
        'Time (s)': pdlp_result['solve_time_sec'],
        'Primal Obj': pdlp_result['primal_obj'],
        'Dual Obj': pdlp_result['dual_obj'],
        'Gap': pdlp_result['duality_gap']
    })

if 'scipy_result' in locals():
    results.append({
        'Problem': 'Small LP',
        'Solver': 'SciPy (HiGHS)',
        'Status': 'optimal' if scipy_result['success'] else scipy_result['status'],
        'Iterations': 'N/A',
        'Time (s)': scipy_result['solve_time_sec'],
        'Primal Obj': scipy_result['primal_obj'],
        'Dual Obj': 'N/A',
        'Gap': 'N/A'
    })

if PDLP_AVAILABLE and 'pdlp_result_large' in locals():
    results.append({
        'Problem': 'Random LP (100x150)',
        'Solver': 'PDLP',
        'Status': pdlp_result_large['status'],
        'Iterations': pdlp_result_large['iterations'],
        'Time (s)': pdlp_result_large['solve_time_sec'],
        'Primal Obj': pdlp_result_large['primal_obj'],
        'Dual Obj': pdlp_result_large['dual_obj'],
        'Gap': pdlp_result_large['duality_gap']
    })

if 'scipy_result_large' in locals():
    results.append({
        'Problem': 'Random LP (100x150)',
        'Solver': 'SciPy (HiGHS)',
        'Status': 'optimal' if scipy_result_large['success'] else scipy_result_large['status'],
        'Iterations': 'N/A',
        'Time (s)': scipy_result_large['solve_time_sec'],
        'Primal Obj': scipy_result_large['primal_obj'],
        'Dual Obj': 'N/A',
        'Gap': 'N/A'
    })

df = pd.DataFrame(results)
display(df)

,Problem,Solver,Status,Iterations,Time (s),Primal Obj,Dual Obj,Gap
0,Small LP,PDLP,optimal,500,0.000301,-9.000007,-8.999999,0.000008
1,Small LP,SciPy (HiGHS),optimal,N/A,0.001066,-9.000000,N/A,N/A
2,Random LP (100x150),PDLP,iteration_limit,50000,0.624781,-2.299150,1748060613636285078380416867940281684497118279...,1748060613636285078380416867940281684497118279...
3,Random LP (100x150),SciPy (HiGHS),The problem is infeasible. (HiGHS Status 8: mo...,N/A,0.001684,NaN,N/A,N/A
